# Model Tuning Notebook

Offline hyperparameter tuning for all models.

**Approach:**
- Hold out `season=2026`
- CV on the remaining seasons: leave-one-season-out as target only
- Compare the held-out test set for each target. Use these hyperparameters, and consider these as error metrics

Targets: `ev_atoi`, `pp_atoi`, `pk_atoi`, `gp_rate`, `evg60`, `eva60`, `ppg60`, `ppa60`

In [ ]:
import os, sys, itertools
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
import xgboost as xgb

sys.path.insert(0, os.path.dirname(os.path.abspath('.')))
from feature_engineering import (
    ALL_TARGETS, FEATURE_SETS,
    build_modeling_frame, training_filter, impute_features, compute_sample_weights,
)

TEST_SEASON = 2026
PROJECTION_YEAR = 2027

frame = build_modeling_frame()
print('Frame rows:', len(frame), '  seasons:', sorted(frame['season'].unique()))

In [ ]:
# Helpers

def fit_predict(target, train_sub, eval_sub, family, hp):
    feats = FEATURE_SETS[target]
    X_tr = train_sub[feats]
    y_tr = train_sub[target].astype(float).values
    w_tr = compute_sample_weights(train_sub, target, hp['decay'], PROJECTION_YEAR)
    X_tr_imp, means = impute_features(X_tr)
    X_ev = eval_sub[feats]
    X_ev_imp, _ = impute_features(X_ev, fitted_means=pd.Series(means))
    if family == 'ridge':
        scaler = StandardScaler()
        Xs_tr = scaler.fit_transform(X_tr_imp.values)
        model = Ridge(alpha=hp['alpha'])
        model.fit(Xs_tr, y_tr, sample_weight=w_tr)
        Xs_ev = scaler.transform(X_ev_imp.values)
        return model.predict(Xs_ev)
    params = dict(n_estimators=hp.get('n_estimators', 200), max_depth=hp.get('max_depth', 3),
                  learning_rate=hp.get('learning_rate', 0.05), subsample=0.8,
                  colsample_bytree=0.8, reg_lambda=1.0, n_jobs=4, verbosity=0)
    model = xgb.XGBRegressor(**params)
    model.fit(X_tr_imp.values, y_tr, sample_weight=w_tr)
    return model.predict(X_ev_imp.values)

def metrics(y_true, y_pred):
    err = y_true - y_pred
    return {
        'mae': float(np.mean(np.abs(err))),
        'rmse': float(np.sqrt(np.mean(err ** 2))),
        'r2': float(1 - np.var(err) / max(np.var(y_true), 1e-9)),
    }

def loso_cv(target, train_frame, family, hp):
    seasons = sorted(train_frame['season'].unique())
    scores = []
    for held in seasons:
        tr = train_frame[train_frame['season'] != held]
        ev = train_frame[train_frame['season'] == held]
        ev = ev[training_filter(ev, target)]
        if ev.empty or tr.empty: continue
        preds = fit_predict(target, tr, ev, family, hp)
        scores.append(metrics(ev[target].astype(float).values, preds)['rmse'])
    return float(np.mean(scores)) if scores else float('nan')

def held_out_test(target, family, hp):
    base = frame[training_filter(frame, target)]
    train = base[base['season'] != TEST_SEASON]
    test = base[base['season'] == TEST_SEASON]
    preds = fit_predict(target, train, test, family, hp)
    return metrics(test[target].astype(float).values, preds)

## Tuning Loop

Grid search

In [ ]:
RIDGE_GRID = {
    'alpha': [0.1, 0.5, 1.0, 2.0, 5.0, 10.0],
    'decay': [0.0, 0.05, 0.10, 0.20],
}
XGB_GRID = {
    'n_estimators': [100, 200, 400],
    'max_depth': [3, 4],
    'learning_rate': [0.03, 0.05, 0.10],
    'decay': [0.0, 0.05, 0.10],
}

def grid_iter(grid):
    keys = list(grid.keys())
    for combo in itertools.product(*[grid[k] for k in keys]):
        yield dict(zip(keys, combo))

results = []
for target in ALL_TARGETS:
    base = frame[training_filter(frame, target)]
    train_pool = base[base['season'] != TEST_SEASON]

    best_ridge = None
    for hp in grid_iter(RIDGE_GRID):
        cv = loso_cv(target, train_pool, 'ridge', hp)
        if best_ridge is None or cv < best_ridge['cv']:
            best_ridge = {**hp, 'cv': cv}

    best_xgb = None
    for hp in grid_iter(XGB_GRID):
        cv = loso_cv(target, train_pool, 'xgb', hp)
        if best_xgb is None or cv < best_xgb['cv']:
            best_xgb = {**hp, 'cv': cv}

    ridge_test = held_out_test(target, 'ridge', best_ridge)
    xgb_test = held_out_test(target, 'xgb', best_xgb)

    winner = 'ridge' if ridge_test['rmse'] <= xgb_test['rmse'] else 'xgb'
    results.append({
        'target': target,
        'winner': winner,
        'ridge_best': best_ridge, 'ridge_test_rmse': ridge_test['rmse'], 'ridge_test_mae': ridge_test['mae'], 'ridge_test_r2': ridge_test['r2'],
        'xgb_best':   best_xgb,   'xgb_test_rmse':   xgb_test['rmse'],   'xgb_test_mae':   xgb_test['mae'],   'xgb_test_r2':   xgb_test['r2'],
    })

results_df = pd.DataFrame(results)
results_df

## `MODEL_CONFIG` block

For `model_training.py`

In [ ]:
lines = ["MODEL_CONFIG = {"]
for r in results:
    if r['winner'] == 'ridge':
        hp = r['ridge_best']
        lines.append(f"    '{r['target']}': {{'family': 'ridge', 'alpha': {hp['alpha']}, 'decay': {hp['decay']}}},")
    else:
        hp = r['xgb_best']
        xgb_p = {k: hp[k] for k in ('n_estimators', 'max_depth', 'learning_rate')}
        lines.append(f"    '{r['target']}': {{'family': 'xgb', 'decay': {hp['decay']}, 'xgb_params': {xgb_p}}},")
lines.append('}')
print('\n'.join(lines))